# Foveation figures — everything `make_protocol_figures.py` draws

None of these needs a forward pass through DeepGaze III. Some explain **what the
transform is** (ch03 §Space-variant blur); the rest explain **what it does**
(ch03 §Training protocol, ch04 §Where the cost falls).

The set is whatever `make_protocol_figures.FIGURES` currently holds — the cell
below prints it with the directory each figure lands in, so this notebook stays
correct as figures are added and moved. `dose_response` and the two `stratified_*`
figures read the production run and are skipped automatically when that data is
absent from a fresh clone; `consensus_method` writes to the consensus-panel
directory rather than to `figs/`.

Not all of them are thesis figures. ch03 includes `gp_pyramid`,
`gp_strength_grid`, `foveation_strength` and `saccade_coverage`; ch04 includes the
two `stratified_*`. The rest
(`gp_eccentricity`, `gp_weights`, `gp_stepvsinterp`, `aliasing_demo`,
`dose_response`, `consensus_method`) are supporting material — cheap to regenerate, and each answers a
question a reader of the code asks, but no chapter includes them.

Every panel is drawn with `tez_deepgaze.foveate_input.Foveation` — the same
object the training and evaluation arms use — so what the thesis shows is what
the experiment ran.

> **Runs on GPU in ~30 s; on CPU it takes minutes.** The deepest pyramid level
> convolves a 145×145 kernel over a full-resolution frame, which is ~50 GFLOP.
> `pick_device()` selects MPS or CUDA automatically when either is available.

In [ ]:
import runpy
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent

from IPython.display import Image, display


def run_script(name, *args):
    """Run scripts/<name> exactly as the command line would.

    These notebooks drive the same code the committed figure came from rather
    than reimplementing the plotting, so a notebook cannot silently disagree
    with what is in results/ and in the thesis.
    """
    sys.argv = [name, *[str(a) for a in args]]
    runpy.run_path(str(REPO / 'scripts' / name), run_name='__main__')


def show(*paths, width=1000):
    for p in paths:
        display(Image(filename=str(REPO / p), width=width))

## PARAMETERS — edit these

In [ ]:
STIM_IDX      = 91                      # the running example carried through ch01/03/04
GRID_STIMS    = [200, 300, 450, 700]    # content spread for the strength grid
ACUITY_CPDS   = [40.0, 20.0, 10.0]      # the three arm cutoffs
PYRAMID_CPD   = 12.0                    # gp_pyramid: below human 40 so the falloff reads
STEPVSINTERP_CPD = 10.0                 # gp_stepvsinterp: strong enough to band visibly

The mechanism figures deliberately use cutoffs *below* the human 40 cyc/deg. At
40 on a 1024×768 frame the falloff is real but too mild to see in print — the
sharp disc alone is 2.96°. The measured arms do not exaggerate; these pictures
do, and say so in their captions.

In [ ]:
sys.path.insert(0, str(REPO / 'scripts'))
import make_protocol_figures as figs

figs.GP_STIM = STIM_IDX
figs.GRID_STIMS = GRID_STIMS
figs.PYRAMID_CPD = PYRAMID_CPD
figs.STEP_CPD = STEPVSINTERP_CPD
figs.OUT.mkdir(parents=True, exist_ok=True)

print('device:', figs.device())
print('ppd   :', figs.PPD, ' e2:', figs.E2_DEG, 'deg')
print('display Nyquist:', figs.NYQ_DISPLAY, 'cyc/deg')

In [ ]:
print('figures this script can produce:')
for name in figs.FIGURES:
    # Not every figure lands in figs.OUT: consensus_method goes to the
    # consensus-panel directory, and the three that read the production run land
    # in the initial tree. FIGURE_DIR is the script's own map.
    out = figs.FIGURE_DIR.get(name, figs.OUT) / f'{name}.png'
    mark = '[on disk]' if out.exists() else '[ missing]'
    print(f"  {mark}  {name:26s} {out.relative_to(figs.ROOT)}")

### The sharp radius

The radii are not fitted — they follow from the viewing geometry alone:

$$e_\text{sharp} = e_2\left(\frac{f_{c0}}{0.5\,\mathrm{ppd}} - 1\right)$$

In [ ]:
for cpd in ACUITY_CPDS:
    r = figs.sharp_radius_deg(cpd)
    print(f'f_c0 = {cpd:5.1f} cpd  ->  sharp disc {r:5.2f} deg = {r * figs.PPD:6.1f} px')

## 1. `gp_pyramid.png` — how the space-variant blur is built

A different blur kernel at every pixel would be expensive, so the image is
foveated by blending an octave-spaced Gaussian pyramid: build the stack once,
compute a per-pixel fractional level $L(e)$, then blend the two bracketing
levels with triangular weights.

In [ ]:
figs.fig_gp_pyramid()
show('results/foveation_mit1003/figs/gp_pyramid.png', width=1300)

## 2. `gp_stepvsinterp.png` — why the blend has to be continuous

Rounding $L(e)$ to the nearest pyramid level gives uniform annuli and visible
rings. Interpolating between the two bracketing levels does not. The right panel
is the same statement as a curve: a staircase against a smooth rise.

In [ ]:
figs.fig_gp_stepvsinterp()
show('results/foveation_mit1003/figs/gp_stepvsinterp.png', width=1300)

## 3. `gp_strength_grid.png` — strength across content

The fovea is pinned to the image centre here purely to isolate *strength* from
*gaze-contingency*. The trained model re-centres on its own current fixation at
every step (ch03 §Gaze-contingent application); a fixed centre is what the
earlier pixel-level foveated networks did.

In [ ]:
figs.GRID_STIMS = GRID_STIMS
figs.fig_gp_strength_grid()
show('results/foveation_mit1003/figs/gp_strength_grid.png', width=1100)

## 4–7. Protocol figures

`aliasing_demo.png` shows what DG3's input halving does to a sharp periphery: it
discards every second pixel with no prefilter, so blurring the periphery also
supplies antialiasing the backbone omits. `foveation_strength.png` derives the
sharp radius — the acuity cutoff crossing the display Nyquist limit — and draws
the disc it implies on a stimulus at true scale. `saccade_coverage.png` puts that
radius against the measured human saccade distribution. `dose_response.png` plots
the measured ΔIG against foveal cutoff, for the gaze-contingent and the
fixed-centre arm, next to the share of saccade targets inside the disc.

> `dose_response.png` reads its ΔIG values from the `DOSE` table in
> `make_protocol_figures.py`. Every row comes from one source — the
> production matrix at `results/foveation_mit1003_initial/test/table.json`, 10 folds
> per arm, reporting epoch 5, test split. It writes into that same tree, which is
> where the cell below reads it from.

In [ ]:
figs.fig_aliasing()
figs.fig_strength()
figs.fig_saccade_coverage()
figs.fig_dose()
show('results/foveation_mit1003/figs/aliasing_demo.png',
     'results/foveation_mit1003/figs/foveation_strength.png',
     'results/foveation_mit1003/figs/saccade_coverage.png',
     'results/foveation_mit1003_initial/figs/dose_response.png', width=1000)

## Regenerate everything from the command line

```bash
.venv/bin/python scripts/make_protocol_figures.py
.venv/bin/python scripts/make_protocol_figures.py --only gp_pyramid gp_strength_grid
```

The run writes `figs/gp_figures.json` recording which stimuli and cutoffs were
used, so the PNGs stay traceable to their parameters.

## Anything else in the set

The sections above walk through the figures one at a time. This regenerates any
remaining entry — including figures added since — and skips the ones whose input
data is not present in this clone.

In [ ]:
walked = {'gp_pyramid', 'gp_stepvsinterp', 'gp_strength_grid',
          'aliasing_demo', 'foveation_strength', 'saccade_coverage',
          'dose_response'}
for name, fn in figs.FIGURES.items():
    if name in walked:
        continue
    out = figs.FIGURE_DIR.get(name, figs.OUT) / f'{name}.png'
    try:
        fn()
    except FileNotFoundError as e:
        # Only the generator's own missing inputs land here. Catching the display
        # as well would report a figure that was written as "skipped".
        print(f'skipped {name}: needs data not in this clone ({e})')
        continue
    print(f'wrote {out.relative_to(figs.ROOT)}')
    show(out, width=1000)